<div class="title-slide">

# Make the Most of What You’ve Got

## Deep Learning Optimization with PyTorch Profiler and Holistic Trace Analysis

<p class="author">Collin Wilson</p>

</div>

# Getting started with `sn-torch-lab`

This talk is given in a Jupyter notebook format. You can download it and all the materials to recreate the demo on GitHub:

```bash
git clone https://github.com/cwilson51/sn-torch-lab.git
```

You can then install the repository as a package in a virtual environment. On Alliance clusters:

```bash
module load StdEnv/2023 python/3.11 scipy-stack/2026a
virtualenv --no-download torch-labENV && source torch-labENV/bin/activate
pip install --no-index --upgrade pip

git clone https://github.com/cwilson51/sn-torch-lab.git
pip install --no-index ./sn-torch-lab
```

<div class="pt-slide">

# What we'll cover today

- PyTorch Profiler revisited
  - Inline profiling
  - Generating traces
- Examining traces and simple optimizations
- Holistic Trace Analysis (HTA)
  - Single GPU
  - Distributed workloads
- Trace analysis with Brine

</div>

<div class="pt-slide">

# PyTorch Profiler

- Built-in module of PyTorch
- Can profile run times, memory and generate detailed traces of PyTorch code
  - See layer-by-layer breakdown of model performance
- Both CPU and GPU profiling
- Code modification required: wrap code with a context manager
- Works for NVIDIA and AMD GPUs; separate tool for Apple MPS
- Today’s live profiles are with the H100 1g.10gb MIG flavour on Nibi

</div>

# Primary components of PyTorch Profiler code

```python
from torch.profiler import ProfilerActivity, profile, record_function, schedule
```

- `ProfilerActivity`: we must pass the profiler a list of activities to profile. Here we'll profile the CPU and GPU if available
- `profile`: the profile object used to create and store the profile during execution
- `record_function`: context manager we can use to label sections of code, e.g. `'training'` and `'inference'` or `'forward-pass'` and `'backward-pass'`
- `schedule`: controls generating profiles over several iterations; we'll see more on this later

<div class="section-slide">

# Working example

## ResNet50 on random inputs

</div>

## Setup

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.profiler import ProfilerActivity, profile, record_function, schedule

from torchvision.models import resnet50
from torchvision.datasets import FakeData
from torchvision import transforms

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    device = torch.device("cuda")
    activities += [ProfilerActivity.CUDA]
    batch_size = 64
else:
    device = torch.device("cpu")
    batch_size = 1

## Model, optimizer and data

In [ ]:
model = resnet50()
model.to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

train_ds = FakeData(
    size=10_000,                 # number of samples
    image_size=(3, 224, 224),
    num_classes=1000,
    transform=transforms.ToTensor(),
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=1, pin_memory=True)

## Run a simple profile

In [ ]:
inputs, labels = next(iter(train_loader))
inputs, labels = inputs.to(device), labels.to(device)

with profile(activities=activities, record_shapes=True) as prof:

    model.train()
    with record_function("model_training"): # record_function can be used to get an aggregate measure of code sections and label them in traces
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with record_function("model_inference"):
        model(inputs)

    prof.step() # !!! Step the profiler !!!

## Results

In [ ]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

Reduced for readability:

<div class="wide">

```
-------------------------------------------------------  ------------  ------------  ------------  ------------
                                                   Name    Self CUDA   Self CUDA %   CUDA time avg   # of Calls
-------------------------------------------------------  ------------  ------------  ------------  ------------
                                         model_training       0.000us         0.00%      88.278ms             1
                                           aten::conv2d       0.000us         0.00%     688.580us           106
                                      aten::convolution       0.000us         0.00%     688.580us           106
                                     aten::_convolution       0.000us         0.00%     688.580us           106
                                aten::cudnn_convolution      72.990ms        22.33%     688.580us           106
autograd::engine::evaluate_function: CudnnBatchNormB...       0.000us         0.00%     736.648us            53
                                CudnnBatchNormBackward0       0.000us         0.00%     736.648us            53
                        aten::cudnn_batch_norm_backward      39.042ms        11.94%     736.648us            53
autograd::engine::evaluate_function: ConvolutionBack...       0.000us         0.00%       1.795ms            53
                                   ConvolutionBackward0       0.000us         0.00%       1.590ms            53
-------------------------------------------------------   ------------  ------------  ------------  ------------
Self CPU time total: 9.422s
Self CUDA time total: 326.916ms
```

</div>

# Exporting traces

- Profiles are most useful as traces that can be analyzed with other tools
- We can export traces as JSON files with `prof.export_chrome_trace("trace.json")`

In [ ]:
prof.export_chrome_trace("example-traces/live/trace.json")

In [ ]:
! ls -lh example-traces/live

# `schedule`

When profiling longer training runs, profiles can get very large. We can capture select iterations using a `schedule`:

```python
my_schedule = schedule(skip_first=10,
                       wait=5,
                       warmup=5,
                       active=5,
                       repeat=1)
```

- The profiler skips the first `skip_first` steps, then waits `wait` steps
- Starting the profile can skew results, so the profiler starts and does a `warmup`
- The profiler is active for `active` steps
- The profiler can repeat a cycle of `wait` + `warmup` + `active` as set by the user with `repeat`

In [ ]:
my_schedule = schedule(skip_first=0,
                       wait=1,
                       warmup=2,
                       active=2,
                       repeat=2)


with profile(activities=activities, schedule=my_schedule, record_shapes=True) as prof:

    model.train()
    for _ in range(10):
    
        inputs, labels = next(iter(train_loader))
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        prof.step()


# Using trace handlers

- As we've seen, the profile only holds one cycle at a time
- We can define a trace handler to print results and/or export traces after each cycle
- The trace handler function is passed to the profiler using the `on_trace_ready` argument

## Example

In [ ]:
sort_by_keyword = "self_" + device + "_time_total"

def trace_handler(p):
    output = p.key_averages().table(sort_by=sort_by_keyword, row_limit=1).splitlines()[-2:]
    print(output)
    p.export_chrome_trace(f"example-traces/live/trace_{p.step_num}.json")

with profile(activities=activities, schedule=my_schedule, record_shapes=True, on_trace_ready=trace_handler) as prof:

    model.train()
    for _ in range(10):
    
        inputs, labels = next(iter(train_loader))
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        prof.step()

In [ ]:
! ls -lh example-traces/live

# Examining traces and simple optimizations

## Traces

- JSON traces are viewable at [ui.perfetto.dev](https://ui.perfetto.dev)
- The traces we'll view are all single-GPU traces with ResNet50 on a synthetic image dataset designed to mimic ImageNet, using an H100 1g.10gb MIG on Nibi
- Traces can be downloaded from [Nextcloud](https://nextcloud.sharcnet.ca/s/smYAE6DynAEQWpn)
- If you've installed this repo, you can also generate the single-GPU examples on your own system or on a cluster with

```bash
torch-lab generate-example-traces
```

# Examining traces and simple optimizations

## Optimizations

- The example traces cover several simple optimizations
  - Serial, `num_workers` unset (defaults to `num_workers=0`)
  - Serial, `num_workers=1`, with pinned memory
  - Serial, `num_workers=1`, compiled with `torch.compile`
  - Threaded, `num_workers=2`, compiled
  - Threaded, `num_workers=2`, compiled with Automatic Mixed Precision
- We don't have time to cover these optimizations in detail, but if you are interested, you can attend my PyTorch course as part of SHARCNET's Intro ARC series at the end of February. Sign up at training.sharcnet.ca!

# Holistic Trace Analysis

- Holistic Trace Analysis (HTA) is a package from the developers of PyTorch for gathering important information from PyTorch Profiler traces
- It is designed for use in Jupyter notebooks, but can be used in scripts as well

<p class="note"><strong>Note:</strong> the PyPI name for HTA has changed to <code>traceinsight</code>, so <code>pip install HolisticTraceAnalysis</code> will only give you up to <code>v0.5.0</code>!</p>

## Let's look at our last model with `hta`

In [ ]:
from hta.trace_analysis import TraceAnalysis
trace_dir = "example-traces/single-gpu/threaded-compiled-amp/"
analyzer = TraceAnalysis(trace_dir=trace_dir)

We noted that the GPU was once again idle after the optimizations; we can check this with `hta`:

In [ ]:
temporal_breakdown_df = analyzer.get_temporal_breakdown()

In [ ]:
print(temporal_breakdown_df['non_compute_time_pctg'].values[0])

HTA lets us look into the idle time further:

In [ ]:
idle_time_df, _ = analyzer.get_idle_time_breakdown()

- `host_wait`: GPU is waiting on the CPU, could be data starved
- `kernel_wait`: overhead associated with launching consecutive kernels on the GPU
- `other`: unattributable idle time

## HTA is nice for single-GPU traces, but really useful for multi-GPU traces

In [ ]:
trace_dir = "example-traces/multi-gpu/1_node_4_gpus/"
analyzer = TraceAnalysis(trace_dir=trace_dir)

In [ ]:
temporal_breakdown_df = analyzer.get_temporal_breakdown()

In [ ]:
idle_time_df, _ = analyzer.get_idle_time_breakdown()

## Kernel breakdown

In [ ]:
analyzer.get_gpu_kernel_breakdown()

- There are many other features of HTA we could use to analyze these traces
- It's still information-dense, and making concrete recommendations for optimizations based on these outputs can take a lot of effort
- Can we use AI to help narrow our focus in making optimizations?

# Trace analysis with Brine

- SHARCNET offers Brine AIaaS to Alliance users
- Raw traces are too large and noisy for LLMs to make use of
- We can run HTA, package the analyses into a prompt and then get recommendations about improvements, or where to look for improvements

# Trace analysis with Brine (and `torch-lab`)

- You've already seen how we can use this presentation's package to generate the example traces
- `torch-lab analyze-traces <trace_dir>` will perform several HTA analyses and package the results into a prompt
- Running with `--no-llm` will only generate the prompt and a small heuristic-based report; you can then use the prompt with another LLM to get the report

In [ ]:
! torch-lab analyze-traces example-traces/single-gpu --no-llm

In [ ]:
! cat hta-report/report.md

In [ ]:
! torch-lab analyze-traces example-traces/single-gpu

In [ ]:
! cat hta-report/report.md

- Traces should be laid out in a well-structured way, with one directory per run
- The package will infer what sort of comparison you are looking for:
  - Single run: suggested improvements
  - Several runs: comparative analysis between the runs and how bottlenecks improve
  - Single-GPU or multi-GPU

In [ ]:
! torch-lab analyze-traces example-traces/multi-gpu/1_node_4_gpus

In [ ]:
! cat hta-report/report.md

In [ ]:
! torch-lab analyze-traces example-traces/multi-gpu

In [ ]:
! cat hta-report/report.md